<a href="https://colab.research.google.com/github/karanamsubbarao/Python-Exercises/blob/main/IISc_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet google-api-python-client google-auth-httplib2

In [ ]:
!pip install google-auth-oauthlib==0.4.6

In [ ]:
# Import all the libraries required for the Project

import base64
import json
import re
import time
from collections import Counter
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from bs4 import BeautifulSoup

from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import pickle
import os


In [ ]:
# ---------------------------------------------------------------------------
# 0. CONFIG — edit these for your situation
# ---------------------------------------------------------------------------
MY_EMAIL = "karanamcloud@gmail.com"  # your own address, used to detect "sent by me"
MANAGER_EMAILS = {"karanamsubbarao@gmail.com"}
DIRECT_REPORT_EMAILS = {"karsubscription@gmail.com","karanamsubbarao@gmail.com"}

# How many days without a reply from you before a thread counts as "you're the
# bottleneck"
BOTTLENECK_DAYS_THRESHOLD = 2

# Threads with more than this many messages get a thread-level summary
LONG_THREAD_MESSAGE_COUNT = 4

In [ ]:
# ---------------------------------------------------------------------------
# 1. LLM SETUP (Hugging Face)
# ---------------------------------------------------------------------------
import torch
from transformers import pipeline

_device = 0 if torch.cuda.is_available() else -1

# Instruct model — needed for structured classification, not just summarization.
# Swap for a bigger model (e.g. Qwen2.5-7B-Instruct) if you have GPU headroom.
classifier_llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device=_device,
    max_new_tokens=150,
    do_sample=False,
)

summarizer_llm = pipeline(
    "text-generation",
    model="facebook/bart-large-cnn",
    device=_device,
)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bias     | UNEXPECTED |  | 
mod

In [ ]:
# ---------------------------------------------------------------------------
# 4. CLASSIFICATION (Action Required / FYI / Noise)
# ---------------------------------------------------------------------------
CLASSIFY_SYSTEM_PROMPT = """You are an email triage assistant. Classify the email into exactly one \
of three labels: "Action Required", "FYI", or "Noise".

- "Action Required": the recipient needs to reply, decide, approve, or do something.
- "FYI": informational, no action needed, but worth reading.
- "Noise": newsletters, automated notifications, marketing, or anything safely ignorable.

Respond with STRICT JSON only, no other text, in this exact format:
{"label": "<one of the three labels>", "reason": "<one short sentence, under 15 words>"}
"""

In [ ]:
#### GMAIL Authentication ####
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def get_gmail_service():
    creds = None
    # Reuse saved token if it exists
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        # run_console works well in Colab since there's no local browser
        creds = flow.run_console()
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)

    return build('gmail', 'v1', credentials=creds)

service = get_gmail_service()



In [ ]:
# ---------------------------------------------------------------------------
# MAIN PIPELINE. Reads the inbox
# ---------------------------------------------------------------------------
def process_inbox(service, query="is:unread", max_results=25):
    print("Building sender frequency map (this scans recent sent/inbox mail)...")
    collaborator_counts = build_frequent_collaborators(service)

    messages = list_messages(service, query=query, max_results=max_results)
    print(f"Found {len(messages)} messages matching query: '{query}'\n")

    results = []

    for m in messages:
        full = get_full_message(service, m["id"])
        headers = full["payload"]["headers"]

        subject = _header(headers, "Subject", "(No subject)")
        sender_raw = _header(headers, "From", "(Unknown sender)")
        sender_email = extract_email_address(sender_raw)
        thread_id = full.get("threadId")

        body = get_email_body(full)
        weight_info = sender_weight(sender_email, collaborator_counts)
        classification = classify_email(subject, sender_email, body, weight_info)
        bottleneck_info = is_bottleneck(service, thread_id)

        # Thread summary only if it's a long thread
        thread_messages = get_thread(service, thread_id)
        thread_summary = None
        if len(thread_messages) > LONG_THREAD_MESSAGE_COUNT:
            thread_summary = summarize_thread(service, thread_id)

        result = {
            "id": m["id"],
            "thread_id": thread_id,
            "subject": subject,
            "from": sender_raw,
            "relationship": weight_info["relationship"],
            "weight": weight_info["weight"],
            "label": classification["label"],
            "reason": classification["reason"],
            "thread_length": len(thread_messages),
            "thread_summary": thread_summary,
            "bottleneck": bottleneck_info,
        }
        results.append(result)

    return results


In [ ]:

# ---------------------------------------------------------------------------
# 1. SENDER WEIGHTING
# ---------------------------------------------------------------------------

def build_frequent_collaborators(service, top_n=25, lookback_query="newer_than:90d"):
    """
    Scans recent sent + received mail to find who you exchange the most email
    with. Returns a Counter of address -> message count.
    """
    counts = Counter()

    for query in (f"in:sent {lookback_query}", f"in:inbox {lookback_query}"):
        messages = list_messages(service, query=query, max_results=200)
        for m in messages:
            full = get_full_message(service, m["id"])
            headers = full["payload"]["headers"]
            frm = extract_email_address(_header(headers, "From"))
            to = extract_email_address(_header(headers, "To"))
            for addr in (frm, to):
                if addr and addr != MY_EMAIL:
                    counts[addr] += 1

    return counts


def sender_weight(sender_email: str, collaborator_counts: Counter) -> dict:
    """
    Returns a relationship weight + label for a sender. Higher weight = more
    important. This directly affects classification (a "FYI" from your manager
    is treated with more urgency than one from a stranger).
    """
    sender_email = sender_email.lower()

    if sender_email in MANAGER_EMAILS:
        return {"weight": 3.0, "relationship": "manager"}
    if sender_email in DIRECT_REPORT_EMAILS:
        return {"weight": 2.5, "relationship": "direct report"}

    freq = collaborator_counts.get(sender_email, 0)
    if freq >= 15:
        return {"weight": 2.0, "relationship": "frequent collaborator"}
    elif freq >= 5:
        return {"weight": 1.5, "relationship": "occasional collaborator"}
    else:
        return {"weight": 1.0, "relationship": "unknown/external"}


In [ ]:
# ---------------------------------------------------------------------------
# 2. GMAIL HELPERS
# ---------------------------------------------------------------------------

def list_messages(service, query="", max_results=25):
    results = service.users().messages().list(
        userId="me", q=query, maxResults=max_results
    ).execute()
    return results.get("messages", [])


def get_full_message(service, msg_id):
    return service.users().messages().get(userId="me", id=msg_id, format="full").execute()


def _header(headers, name, default=""):
    return next((h["value"] for h in headers if h["name"].lower() == name.lower()), default)


def extract_email_address(header_value: str) -> str:
    """Pulls 'a@b.com' out of 'Name <a@b.com>'."""
    match = re.search(r"[\w\.\-\+]+@[\w\.\-]+", header_value or "")
    return match.group(0).lower() if match else ""


def get_email_body(message) -> str:
    payload = message["payload"]

    def extract_text(parts):
        text = ""
        for part in parts:
            mime_type = part.get("mimeType", "")
            body_data = part.get("body", {}).get("data")
            if mime_type == "text/plain" and body_data:
                text += base64.urlsafe_b64decode(body_data).decode("utf-8", errors="ignore")
            elif mime_type == "text/html" and body_data and not text:
                html = base64.urlsafe_b64decode(body_data).decode("utf-8", errors="ignore")
                text += BeautifulSoup(html, "html.parser").get_text(separator=" ")
            elif "parts" in part:
                text += extract_text(part["parts"])
        return text

    if "parts" in payload:
        return extract_text(payload["parts"])
    elif payload.get("body", {}).get("data"):
        return base64.urlsafe_b64decode(payload["body"]["data"]).decode("utf-8", errors="ignore")
    return ""


def get_thread(service, thread_id):
    thread = service.users().threads().get(userId="me", id=thread_id, format="full").execute()
    return thread.get("messages", [])


In [ ]:
### -------------- #########
### Classify Email #########
### -------------- #########
def classify_email(subject: str, sender_email: str, body: str, weight_info: dict) -> dict:
    truncated_body = body.strip()[:1500]

    prompt = (
        f"Sender: {sender_email} (relationship: {weight_info['relationship']})\n"
        f"Subject: {subject}\n"
        f"Body:\n{truncated_body}\n"
    )

    raw = _chat(prompt, system=CLASSIFY_SYSTEM_PROMPT)

    # Be defensive — small instruct models sometimes wrap JSON in extra text
    json_match = re.search(r"\{.*\}", raw, re.DOTALL)
    if json_match:
        try:
            parsed = json.loads(json_match.group(0))
            label = parsed.get("label", "FYI")
            reason = parsed.get("reason", "").strip()
            if label not in ("Action Required", "FYI", "Noise"):
                label = "FYI"
            return {"label": label, "reason": reason or "(no reason given)"}
        except json.JSONDecodeError:
            pass

    return {"label": "FYI", "reason": "(could not parse model output)"}



In [ ]:
# ---------------------------------------------------------------------------
# THREAD SUMMARIZATION
# ---------------------------------------------------------------------------

def summarize_thread(service, thread_id: str) -> str:
    messages = get_thread(service, thread_id)

    combined = []
    for m in messages:
        headers = m["payload"]["headers"]
        sender = _header(headers, "From")
        body = get_email_body(m).strip()
        if body:
            combined.append(f"{sender} wrote: {body[:800]}")

    full_text = "\n\n".join(combined)[:4000]
    if not full_text.strip():
        return "(No content to summarize)"

    try:
        summary = summarizer_llm(full_text, max_length=120, min_length=30, do_sample=False)
        return summary[0]["summary_text"]
    except Exception as e:
        return f"(Could not summarize thread: {e})"


In [ ]:

# ---------------------------------------------------------------------------
# BOTTLENECK DETECTION
# ---------------------------------------------------------------------------

def is_bottleneck(service, thread_id: str) -> dict:
    """
    You're the bottleneck if the LAST message in the thread was sent TO you
    (not by you), and it's been more than BOTTLENECK_DAYS_THRESHOLD days since
    then without a reply from you.
    """
    messages = get_thread(service, thread_id)
    if not messages:
        return {"bottleneck": False}

    last_msg = messages[-1]
    headers = last_msg["payload"]["headers"]
    last_sender = extract_email_address(_header(headers, "From"))

    if last_sender == MY_EMAIL:
        return {"bottleneck": False}  # you already replied last

    date_str = _header(headers, "Date")
    try:
        last_date = parsedate_to_datetime(date_str)
        if last_date.tzinfo is None:
            last_date = last_date.replace(tzinfo=timezone.utc)
    except Exception:
        return {"bottleneck": False}

    days_waiting = (datetime.now(timezone.utc) - last_date).days

    return {
        "bottleneck": days_waiting >= BOTTLENECK_DAYS_THRESHOLD,
        "days_waiting": days_waiting,
        "waiting_on": last_sender,
    }


In [ ]:
def _chat(prompt: str, system: str = "") -> str:
    """Helper to run the instruct model with a chat template and return raw text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    out = classifier_llm(messages)
    # pipeline with chat-style input returns generated_text as a list of turns
    generated = out[0]["generated_text"]
    if isinstance(generated, list):
        return generated[-1]["content"]
    return str(generated)


In [ ]:
def print_triage_report(results):
    # Sort: Action Required first, then by sender weight descending
    label_priority = {"Action Required": 0, "FYI": 1, "Noise": 2}
    results_sorted = sorted(
        results, key=lambda r: (label_priority.get(r["label"], 3), -r["weight"])
    )

    for r in results_sorted:
        flag = "🔴" if r["bottleneck"].get("bottleneck") else "  "
        print(f"{flag} [{r['label']:15s}] {r['subject']}")
        print(f"     From: {r['from']}  ({r['relationship']}, weight {r['weight']})")
        print(f"     Reason: {r['reason']}")
        if r["bottleneck"].get("bottleneck"):
            print(
                f"     ⚠️  Waiting on you for {r['bottleneck']['days_waiting']} days "
                f"(from {r['bottleneck']['waiting_on']})"
            )
        if r["thread_summary"]:
            print(f"     Thread summary ({r['thread_length']} messages): {r['thread_summary']}")
        print("-" * 70)

In [ ]:
# ---------------------------------------------------------------------------
# Usage in Colab:
# ---------------------------------------------------------------------------
results = process_inbox(service, query="is:unread", max_results=25)
print_triage_report(results)